# ASL Preprocessing

***Things I changed for this setup:***

- added a "holdout" test set of 3 participants, never seen during CV so we can be sure it is generalizing the signs and not learning based on the participant.
- then the the remaining 18 participants are going to be split into 5 GroupKFold CV folds so we can tune the hyper parameters to achieve the highest accuracy

Also this will save the data so this is the only time this is needed to be run.

In [9]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

DATA_DIR      = r'D:\MLproject\asl-signs'
TRAIN_CSV     = os.path.join(DATA_DIR, 'train.csv')
OUTPUT_DIR    = r'D:\MLproject\code\processed_data'
TARGET_FRAMES = 30
MIN_FRAMES    = 5
# Caden's random seed
RANDOM_SEED   = 69
N_HOLDOUT_PARTICIPANTS = 3
N_CV_FOLDS    = 5

# 50 signs copied from original preproccessing
SELECTED_SIGNS = [
    'bird', 'fish', 'duck', 'frog', 'alligator', 'cat', 'dog', 'cow',
    'pig', 'tiger', 'lion', 'horse', 'wolf', 'bee', 'owl', 'goose',
    'jump', 'dance', 'blow', 'drink', 'drop', 'find', 'give', 'make',
    'cry', 'read', 'cut', 'hide', 'fall', 'ride',
    'yes', 'no', 'finish', 'open', 'close', 'up', 'down', 'fast',
    'quiet', 'wait', 'now', 'later', 'every', 'same', 'any',
    'pizza', 'boat', 'airplane', 'rain', 'snow'
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')
print(f'Classes: {len(SELECTED_SIGNS)}')

Output dir: D:\MLproject\code\processed_data
Classes: 50


In [10]:
# ── Load and filter train.csv ─────────────────────────────────────────────────
train = pd.read_csv(TRAIN_CSV)
print(f'Total clips in dataset: {len(train)}')
print(f'Unique signs in dataset: {train["sign"].nunique()}')
print(f'Unique participants: {train["participant_id"].nunique()}')

# This will show us the change in amount of size, which is just the number of "videos" or mediapipe landmarks we have from each video
train = train[train['sign'].isin(SELECTED_SIGNS)].reset_index(drop=True)
print(f'\nAfter filtering to {len(SELECTED_SIGNS)} selected signs:')
print(f'  Clips: {len(train)}')
print(f'  Participants: {train["participant_id"].nunique()}')
print(f'  Avg clips/sign: {len(train)/train["sign"].nunique():.1f}')

Total clips in dataset: 94477
Unique signs in dataset: 250
Unique participants: 21

After filtering to 50 selected signs:
  Clips: 18907
  Participants: 21
  Avg clips/sign: 378.1


In [11]:
# The Google dataset has exactly 543 landmarks per frame, which are distributed - face: 0-467, pose: 468-500, left_hand: 501-521, right_hand: 522-542
ROWS_PER_FRAME = 543
LEFT_HAND_IDX  = slice(501, 522)
RIGHT_HAND_IDX = slice(522, 543)

def load_parquet(path):
    df = pd.read_parquet(path, columns=['x', 'y', 'z'])
    n_frames = int(len(df) / ROWS_PER_FRAME)
    if n_frames == 0:
        return None

    data = df.values.reshape(n_frames, ROWS_PER_FRAME, 3).astype(np.float32)
    left  = data[:, LEFT_HAND_IDX,  :]
    right = data[:, RIGHT_HAND_IDX, :]
    # Only using the hand landmarks for our tests thorughout each model
    seq = np.concatenate([left[:, :, 0],  left[:, :, 1],  left[:, :, 2], right[:, :, 0], right[:, :, 1], right[:, :, 2],], axis=1)
    seq = np.nan_to_num(seq, nan=0.0)

    # don't want to use meaningless frames so remove when both hands are missing or no landmarks are present
    left_missing  = np.all(seq[:, 0:63]   == 0, axis=1)
    right_missing = np.all(seq[:, 63:126] == 0, axis=1)
    valid = ~(left_missing & right_missing)
    seq = seq[valid]
    # checking for at least 5 frames
    if len(seq) < MIN_FRAMES:
        return None
    return seq


def interpolate(seq, target=TARGET_FRAMES):
    # We want to have a common amount of frames
    T = len(seq)
    if T == target:
        return seq
    x_old = np.linspace(0, 1, T)
    x_new = np.linspace(0, 1, target)
    out = np.zeros((target, seq.shape[1]), dtype=np.float32)
    for i in range(seq.shape[1]):
        out[:, i] = np.interp(x_new, x_old, seq[:, i])
    return out


def normalize(seq):
    # Originally we were using the mid point of the wrist to normalize the positions of the hands so we could still pull meaningful data, but
    # realized this took away from the depth part of signs, so switched to calculating a distance between the hands
    seq = seq.copy()
    for f in range(len(seq)):
        lx, ly, lz = seq[f, 0], seq[f, 21], seq[f, 42]
        rx, ry, rz = seq[f, 63], seq[f, 84], seq[f, 105]
        left_ok  = not (lx == 0 and ly == 0 and lz == 0)
        right_ok = not (rx == 0 and ry == 0 and rz == 0)
        if not left_ok and not right_ok:
            continue
        if left_ok and right_ok:
            rx_, ry_, rz_ = (lx+rx)/2, (ly+ry)/2, (lz+rz)/2
        elif left_ok:
            rx_, ry_, rz_ = lx, ly, lz
        else:
            rx_, ry_, rz_ = rx, ry, rz
        seq[f, 0:21]    -= rx_
        seq[f, 21:42]   -= ry_
        seq[f, 42:63]   -= rz_
        seq[f, 63:84]   -= rx_
        seq[f, 84:105]  -= ry_
        seq[f, 105:126] -= rz_
    return seq

sample_path = os.path.join(DATA_DIR, train['path'].iloc[0])
s = load_parquet(sample_path)
print(f'Sample shape: {s.shape}')
s = interpolate(s)
s = normalize(s)
print(f'After interp+norm: {s.shape}')
print(f'NaN count: {np.isnan(s).sum()}')

Sample shape: (23, 126)
After interp+norm: (30, 126)
NaN count: 0


In [ ]:
# preprocessing starts here
X_all, y_all, groups_all = [], [], []
skipped = 0

for _, row in tqdm(train.iterrows(), total=len(train)):
    path = os.path.join(DATA_DIR, row['path'])
    seq = load_parquet(path)
    if seq is None:
        skipped += 1
        continue
    seq = interpolate(seq)
    seq = normalize(seq)
    X_all.append(seq)
    y_all.append(row['sign'])
    groups_all.append(row['participant_id'])

X_all = np.array(X_all, dtype=np.float32)
y_all = np.array(y_all)
groups_all = np.array(groups_all)

print(f'\nFinal: {X_all.shape}, labels {y_all.shape}, groups {groups_all.shape}')
print(f'Skipped: {skipped}')
print(f'Unique participants: {len(np.unique(groups_all))}')

 61%|██████████████████████████████████████████████▍                             | 11544/18907 [04:40<03:00, 40.69it/s]

In [5]:
# labels
le = LabelEncoder()
y_enc = le.fit_transform(y_all)
print(f'Classes: {len(le.classes_)}')
print(f'Class names: {list(le.classes_)}')

Classes: 50
Class names: [np.str_('airplane'), np.str_('alligator'), np.str_('any'), np.str_('bee'), np.str_('bird'), np.str_('blow'), np.str_('boat'), np.str_('cat'), np.str_('close'), np.str_('cow'), np.str_('cry'), np.str_('cut'), np.str_('dance'), np.str_('dog'), np.str_('down'), np.str_('drink'), np.str_('drop'), np.str_('duck'), np.str_('every'), np.str_('fall'), np.str_('fast'), np.str_('find'), np.str_('finish'), np.str_('fish'), np.str_('frog'), np.str_('give'), np.str_('goose'), np.str_('hide'), np.str_('horse'), np.str_('jump'), np.str_('later'), np.str_('lion'), np.str_('make'), np.str_('no'), np.str_('now'), np.str_('open'), np.str_('owl'), np.str_('pig'), np.str_('pizza'), np.str_('quiet'), np.str_('rain'), np.str_('read'), np.str_('ride'), np.str_('same'), np.str_('snow'), np.str_('tiger'), np.str_('up'), np.str_('wait'), np.str_('wolf'), np.str_('yes')]


### Mostly Claude from here down, never used the unique participants setup so basically just copy and pasted to get moving onto the RF ###

In [6]:
# setting up the holdout test with the 3 participants that won't be in the training data
unique_participants = np.unique(groups_all)
print(f'Total participants: {len(unique_participants)}')

# splits evenly
holdout_frac = N_HOLDOUT_PARTICIPANTS / len(unique_participants)
gss = GroupShuffleSplit(n_splits=1, test_size=holdout_frac, random_state=RANDOM_SEED)
dev_idx, test_idx = next(gss.split(X_all, y_enc, groups=groups_all))

X_dev, y_dev, groups_dev = X_all[dev_idx],  y_enc[dev_idx],  groups_all[dev_idx]
X_test, y_test, groups_test = X_all[test_idx], y_enc[test_idx], groups_all[test_idx]

print(f'\nDev set:  {X_dev.shape}, {len(np.unique(groups_dev))} participants')
print(f'Test set: {X_test.shape}, {len(np.unique(groups_test))} participants')
print(f'Test participants: {np.unique(groups_test)}')

# double check to make sure none are contaminated
assert len(set(groups_dev) & set(groups_test)) == 0, 'participant leakage!'

Total participants: 21

Dev set:  (15924, 30, 126), 18 participants
Test set: (2924, 30, 126), 3 participants
Test participants: [37055 49445 55372]


In [ ]:
# 5 cross validation folds to hep with hyper paramater tuning
gkf = GroupKFold(n_splits=N_CV_FOLDS)
fold_indices = []
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_dev, y_dev, groups=groups_dev)):
    fold_indices.append((tr_idx, va_idx))
    tr_participants = set(groups_dev[tr_idx])
    va_participants = set(groups_dev[va_idx])
    assert not (tr_participants & va_participants), f'fold {fold} leaks!'
    print(f'Fold {fold+1}: train={len(tr_idx)} clips / {len(tr_participants)} ppl, '
          f'val={len(va_idx)} clips / {len(va_participants)} ppl')

In [8]:
# ── Save everything ───────────────────────────────────────────────────────────
# The downstream notebooks just load these files.

np.savez_compressed(
    os.path.join(OUTPUT_DIR, 'dev.npz'),
    X=X_dev, y=y_dev, groups=groups_dev
)
np.savez_compressed(
    os.path.join(OUTPUT_DIR, 'test.npz'),
    X=X_test, y=y_test, groups=groups_test
)
# Save fold indices so both models use identical splits
np.savez(
    os.path.join(OUTPUT_DIR, 'cv_folds.npz'),
    **{f'fold_{i}_train': tr for i, (tr, _) in enumerate(fold_indices)},
    **{f'fold_{i}_val':   va for i, (_, va) in enumerate(fold_indices)},
)
# Save label encoder classes
np.save(os.path.join(OUTPUT_DIR, 'classes.npy'), le.classes_)

print('Saved:')
for f in ['dev.npz', 'test.npz', 'cv_folds.npz', 'classes.npy']:
    p = os.path.join(OUTPUT_DIR, f)
    print(f'  {f}: {os.path.getsize(p)/1024/1024:.1f} MB')

Saved:
  dev.npz: 155.6 MB
  test.npz: 26.5 MB
  cv_folds.npz: 0.6 MB
  classes.npy: 0.0 MB
